# Learn PyDESeq2 Step by Step

This notebook walks through every step of PyDESeq2 using a **tiny fake dataset**  
so you can see exactly what each function does.

Run each cell one at a time (Shift+Enter) and read the output!

## Step 0: Create Fake Data

We're making up a small dataset:
- **6 samples**: 3 leaf, 3 root
- **10 genes**: some will be "differentially expressed" on purpose

This mimics the structure of your real `gene_count_matrix.tsv` and `sample_metadata.tsv`.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

samples = ["leaf_1", "leaf_2", "leaf_3", "root_1", "root_2", "root_3"]

gene_names = [
    "gene_leaf_high_1",   # intentionally high in leaf
    "gene_leaf_high_2",   # intentionally high in leaf
    "gene_root_high_1",   # intentionally high in root
    "gene_root_high_2",   # intentionally high in root
    "gene_same_1",        # similar in both (NOT differentially expressed)
    "gene_same_2",        # similar in both
    "gene_low_counts",    # very low counts (noisy)
    "gene_medium",        # moderate expression
    "gene_high_both",     # high in both tissues
    "gene_variable",      # high variability between replicates
]

# Build the count matrix by hand so we KNOW what to expect
#                        leaf_1  leaf_2  leaf_3  root_1  root_2  root_3
count_data = np.array([
    [  800,   750,   820,    50,    60,    45],  # gene_leaf_high_1: ~15x higher in leaf
    [  500,   480,   520,    30,    25,    35],  # gene_leaf_high_2: ~16x higher in leaf
    [   40,    35,    50,   900,   850,   920],  # gene_root_high_1: ~20x higher in root
    [   20,    25,    30,   600,   580,   620],  # gene_root_high_2: ~24x higher in root
    [  300,   310,   290,   280,   300,   310],  # gene_same_1: about equal
    [  200,   190,   210,   195,   205,   200],  # gene_same_2: about equal
    [    2,     5,     1,     3,     4,     2],  # gene_low_counts: too few reads to trust
    [  150,   140,   160,   100,   110,    90],  # gene_medium: slightly higher in leaf
    [ 1000,   980,  1020,   950,   970,   960],  # gene_high_both: high everywhere, similar
    [  100,   500,    50,   200,   400,    80],  # gene_variable: noisy replicates
])

# Create the count matrix DataFrame (genes as rows, samples as columns)
count_matrix = pd.DataFrame(count_data, index=gene_names, columns=samples)

print("=" * 60)
print("COUNT MATRIX (raw read counts)")
print("=" * 60)
print(f"Shape: {count_matrix.shape[0]} genes x {count_matrix.shape[1]} samples")
print()
print(count_matrix)
print()
print("Each number = how many times that gene was detected in that sample")

In [ ]:
# Create metadata (tells PyDESeq2 which samples are leaf vs root)
metadata = pd.DataFrame({
    "sample": samples,
    "condition": ["L", "L", "L", "R", "R", "R"]
})
metadata = metadata.set_index("sample")

print("=" * 60)
print("METADATA (which group each sample belongs to)")
print("=" * 60)
print()
print(metadata)
print()
print("This is the 'condition' column that CONTRAST_FACTOR points to.")
print("The VALUES here ('L' and 'R') must match CONTRAST_A and CONTRAST_B.")

## Step 1: Create the DeseqDataSet

This is the main object that holds your data. PyDESeq2 wants:
- **counts**: samples as ROWS, genes as COLUMNS (transposed from how you usually see it)
- **metadata**: one row per sample, with the grouping column

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# PyDESeq2 wants samples as ROWS, genes as COLUMNS
# Our count_matrix has genes as rows, so we transpose it
counts_for_pydeseq = count_matrix.T  # .T = transpose

print("BEFORE transpose (your count matrix):")
print(f"  Rows = genes ({count_matrix.shape[0]}), Columns = samples ({count_matrix.shape[1]})")
print()
print("AFTER transpose (what PyDESeq2 wants):")
print(f"  Rows = samples ({counts_for_pydeseq.shape[0]}), Columns = genes ({counts_for_pydeseq.shape[1]})")
print()
print(counts_for_pydeseq)

In [ ]:
# Create the DeseqDataSet object
dds = DeseqDataSet(
    counts=counts_for_pydeseq,
    metadata=metadata,
    design_factors="condition",  # which column in metadata to use
    refit_cooks=True,
    n_cpus=1
)

print("DeseqDataSet created!")
print(f"  Type: {type(dds)}")
print(f"  Samples: {dds.obs.index.tolist()}")
print(f"  Genes: {dds.var.index.tolist()}")
print(f"  Conditions: {dds.obs['condition'].tolist()}")

## Step 2: Size Factor Estimation (Normalization)

**Why?** Different samples may have been sequenced deeper than others.  
A sample with 2x more total reads will have ~2x the counts for EVERY gene,  
even genes that aren't actually more expressed.

Size factors correct for this. A size factor of 1.2 means that sample had  
20% more reads than average, so its counts will be divided by 1.2.

In [ ]:
# Before normalization, look at total counts per sample
print("TOTAL COUNTS PER SAMPLE (before normalization):")
print()
for sample in samples:
    total = count_matrix[sample].sum()
    print(f"  {sample}: {total:,} reads")

print()
print("If these numbers are very different, normalization matters a lot.")
print("(Our fake data has similar totals, so size factors will be close to 1.0)")

In [ ]:
# Run size factor estimation
dds.fit_size_factors()

print("SIZE FACTORS (normalization values):")
print()
for sample, sf in zip(dds.obs.index, dds.obsm["size_factors"]):
    print(f"  {sample}: {sf:.4f}")

print()
print("Interpretation:")
print("  > 1.0 = sample had MORE reads than average (counts scaled DOWN)")
print("  < 1.0 = sample had FEWER reads than average (counts scaled UP)")
print("  = 1.0 = sample is right at the average")

## Step 3: Dispersion Estimation (Measuring Noise)

**Why?** Your 3 leaf replicates won't have identical counts -- there's natural biological variation.  
PyDESeq2 needs to know how much variation is "normal" for each gene so it can tell  
the difference between real expression changes and random noise.

**Dispersion** = how spread out the replicates are.  
- High dispersion = very noisy gene (hard to trust fold changes)
- Low dispersion = consistent replicates (fold changes are reliable)

In [ ]:
# Run dispersion estimation
dds.fit_genewise_dispersions()

print("GENE-WISE DISPERSIONS (per-gene noise estimates):")
print()

dispersions = dds.varm["genewise_dispersions"]
for gene, disp in zip(dds.var.index, dispersions):
    print(f"  {gene:25s}: {disp:.6f}")

print()
print("Higher number = noisier gene = harder to call significant")
print("Notice 'gene_variable' likely has high dispersion (noisy replicates)")
print("And 'gene_same_1' likely has low dispersion (consistent replicates)")

In [ ]:
# Fit the dispersion trend and shrink dispersions
dds.fit_dispersion_trend()
dds.fit_dispersion_prior()
dds.fit_MAP_dispersions()

print("Dispersion shrinkage complete!")
print()
print("PyDESeq2 'borrows strength' across genes:")
print("  - Genes with few counts get their dispersion pulled toward the trend")
print("  - This prevents unreliable estimates from low-count genes")

## Step 4: Fit the Statistical Model

Now PyDESeq2 fits a **negative binomial model** to each gene.  
This calculates the log2 fold change (how different the gene is between conditions).

In [ ]:
dds.fit_LFC()
dds.calculate_cooks()

print("Model fitting complete!")
print()
print("Now PyDESeq2 knows:")
print("  - How much each gene differs between leaf and root (fold change)")
print("  - How noisy each gene is (dispersion)")
print("  - How to normalize for sequencing depth (size factors)")
print()
print("Next: statistical testing to get p-values!")

## Step 5: Statistical Testing (The Contrast!)

This is where you set **A vs B** (your contrast).  

Remember:  
- `contrast = ("condition", "L", "R")` means: **leaf vs root, with root as baseline**  
- Positive log2FC = higher in **leaf** (A)  
- Negative log2FC = higher in **root** (B)

In [ ]:
# Set up the contrast: (factor_column, A, B)
# A = numerator (what you're measuring)
# B = denominator (baseline)
stat_res = DeseqStats(
    dds,
    contrast=("condition", "L", "R"),  # leaf vs root, root is baseline
    n_cpus=1
)

# Run the Wald test (calculates p-values)
stat_res.summary()

print("\nStatistical testing complete!")

## Step 6: Look at the Results!

This is the final output table -- the same thing your real pipeline saves as  
`pydeseq2_results_UNFILTERED.tsv`.

In [ ]:
results = stat_res.results_df

print("=" * 80)
print("FULL RESULTS TABLE")
print("=" * 80)
print()
print(results.to_string())
print()
print("\nCOLUMN MEANINGS:")
print("  baseMean       = average expression across ALL samples")
print("  log2FoldChange = how different (positive = higher in LEAF)")
print("  lfcSE          = uncertainty in the fold change")
print("  stat           = test statistic (log2FC / lfcSE)")
print("  pvalue         = raw p-value")
print("  padj           = adjusted p-value (THIS is what you filter on)")

In [ ]:
print("=" * 80)
print("INTERPRETING EACH GENE")
print("=" * 80)
print()

for gene in results.index:
    row = results.loc[gene]
    lfc = row["log2FoldChange"]
    padj = row["padj"]
    basemean = row["baseMean"]
    
    # Determine significance
    if pd.isna(padj):
        sig = "NO RESULT (filtered out)"
    elif padj < 0.05 and abs(lfc) > 1:
        direction = "LEAF" if lfc > 0 else "ROOT"
        sig = f"*** SIGNIFICANT - higher in {direction} ***"
    elif padj < 0.05:
        sig = "significant but small fold change"
    else:
        sig = "NOT significant (padj >= 0.05)"
    
    print(f"{gene:25s} | log2FC={lfc:+7.2f} | padj={padj if pd.notna(padj) else 'NA':>10} | {sig}")

## Try It Yourself!

Now experiment! Some things to try:

1. **Flip the contrast** -- change `("condition", "L", "R")` to `("condition", "R", "L")` and re-run. Watch the log2FC signs flip!

2. **Change the fake data** -- go back to Step 0 and make a gene have 1000 in leaf and 999 in root. Is it significant? (No -- the difference is tiny relative to the mean.)

3. **Add more noise** -- change `gene_same_1` to have counts like `[100, 500, 300, 280, 300, 310]`. More variability in replicates = harder to detect real differences.

In [ ]:
# YOUR PLAYGROUND - try things here!
# For example, filter for significant genes:

significant = results[(results["padj"] < 0.05) & (abs(results["log2FoldChange"]) > 1)]

print(f"Significant genes (padj < 0.05 and |log2FC| > 1): {len(significant)}")
print()
print(significant[["log2FoldChange", "padj"]])